In [1]:
import pandas as pd

df = pd.read_csv("final_integrated_dataset.csv")

# Keep one monthly traffic row per month
monthly = df[["YEAR", "MONTH", "TOTAL_TRAFFIC"]].drop_duplicates().copy()

# Create a proper date
monthly["date"] = pd.to_datetime(monthly[["YEAR", "MONTH"]].assign(DAY=1))
monthly = monthly.sort_values("date").reset_index(drop=True)

# Create lag features
monthly["lag_1"] = monthly["TOTAL_TRAFFIC"].shift(1)
monthly["lag_2"] = monthly["TOTAL_TRAFFIC"].shift(2)
monthly["lag_3"] = monthly["TOTAL_TRAFFIC"].shift(3)
monthly["lag_6"] = monthly["TOTAL_TRAFFIC"].shift(6)
monthly["lag_12"] = monthly["TOTAL_TRAFFIC"].shift(12)

# Optional rolling averages
monthly["roll_3"] = monthly["TOTAL_TRAFFIC"].shift(1).rolling(3).mean()
monthly["roll_6"] = monthly["TOTAL_TRAFFIC"].shift(1).rolling(6).mean()

# Drop rows with missing lag values
ts_model = monthly.dropna().copy()

# Keep clean forecasting dataset
ts_model = ts_model[[
    "YEAR", "MONTH",
    "lag_1", "lag_2", "lag_3", "lag_6", "lag_12",
    "roll_3", "roll_6",
    "TOTAL_TRAFFIC"
]]

print(ts_model.shape)
ts_model.head()

ts_model.to_csv("final_integrated_model3_timeseries.csv", index=False)

(137, 10)


In [2]:
import pandas as pd

ts_model = pd.read_csv("final_integrated_model3_timeseries.csv")

# Keep order as time order
train = ts_model.iloc[:-12].copy()   # all except last 12 months
test = ts_model.iloc[-12:].copy()    # last 12 months

print("Train:", train.shape)
print("Test:", test.shape)

train.to_csv("model3_train.csv", index=False)
test.to_csv("model3_test.csv", index=False)

Train: (125, 10)
Test: (12, 10)
